# Drought ensemble — finished runs

Plots completed `droughts/` members on **potomac2** and **wolf2**.

Sequence layout: 40 yr average spinup → drought (`dry`) → 5 yr recovery. Drought begins at year 40.

Finished members:
- both domains: `short_baseline`, `1_year_drought`, `3_year_drought`, `10_year_drought`
- wolf2: `50_year_drought` (long `baseline` still running; included automatically when ready)

Uses `utils.read_storage_outlet_series` (year-by-year, storage + outlet only, coarse time sampling).
Figures also go under `analysis/figures/`.

In [7]:
try:
    from drought_ensemble.analysis.paper_figures import utils
    from drought_ensemble.classes import Domain
except ImportError:
    import sys
    from pathlib import Path

    def _find_project_root() -> Path:
        for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
            if (candidate / "classes" / "Domain.py").is_file():
                return candidate
            nested = candidate / "drought-ensemble"
            if (nested / "classes" / "Domain.py").is_file():
                return nested
        raise ImportError(
            "Install with `pip install -e .` from drought-ensemble/, "
            "or run this notebook from inside the project tree."
        )

    _root = _find_project_root()
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))

    from analysis.paper_figures import utils
    from classes import Domain

In [8]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ENSEMBLE = "droughts"
DOMAINS = ["potomac2", "wolf2"]
PROJECT_ROOT = Path("/glade/derecho/scratch/bwest/drought-ensemble")

FINISHED = {
    "potomac2": [
        "short_baseline",
        "1_year_drought",
        "3_year_drought",
        "10_year_drought",
    ],
    "wolf2": [
        "short_baseline",
        "1_year_drought",
        "3_year_drought",
        "10_year_drought",
        "50_year_drought",
    ],
}

# Pull in long baseline / 50yr if processed outputs exist
for domain_name in DOMAINS:
    for member in ("baseline", "50_year_drought"):
        loc = (
            PROJECT_ROOT
            / "domains"
            / domain_name
            / "processed_full_runs"
            / ENSEMBLE
            / member
            / "file_locations.json"
        )
        if loc.is_file() and member not in FINISHED[domain_name]:
            FINISHED[domain_name].append(member)
            print(f"including {domain_name}/{member}")

SHORT_DROUGHTS = ["1_year_drought", "3_year_drought", "10_year_drought"]

SPINUP_YEARS = 40
SPINUP_YEARS_TO_KEEP = 3
PLOT_START_YEAR = SPINUP_YEARS - SPINUP_YEARS_TO_KEEP  # 37
DROUGHT_START_YEAR = SPINUP_YEARS  # 40

# Coarse sampling keeps long drought records fast (~quarterly)
INTERVAL = 2190

FIG_DIR = PROJECT_ROOT / "analysis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

outlets = {}
for domain_name in DOMAINS:
    domain = Domain(domain_name, full_config_file_path_given=False)
    outlets[domain_name] = (domain.outlet_x, domain.outlet_y)
    print(domain_name, "outlet", outlets[domain_name])

potomac2 outlet (66, 135)
wolf2 outlet (18, 21)


In [ ]:
def mark_drought(ax, *, label=False):
    ax.axvline(
        DROUGHT_START_YEAR,
        color="k",
        ls="--",
        lw=1,
        label="drought starts" if label else None,
    )
    ax.grid(True, alpha=0.3)


def short_label(member):
    return member.replace("_drought", "").replace("_", " ")


def delta_from_onset(da):
    t = da.time.values
    i0 = int(np.argmin(np.abs(t - DROUGHT_START_YEAR)))
    return da - float(da.isel(time=i0).values)


series = {}
for domain_name, members in FINISHED.items():
    series[domain_name] = {}
    ox, oy = outlets[domain_name]
    for member in members:
        print(f"Loading {domain_name}/{member}…", flush=True)
        ds = utils.read_storage_outlet_series(
            ENSEMBLE,
            member,
            domain_name,
            start_year=PLOT_START_YEAR,
            interval=INTERVAL,
            outlet_x=ox,
            outlet_y=oy,
        )
        series[domain_name][member] = ds
        print(
            f"  time {float(ds.time.min()):.2f} → {float(ds.time.max()):.2f} yr "
            f"({ds.sizes['time']} samples)",
            flush=True,
        )

Loading potomac2/short_baseline…
  time 37.00 → 54.75 yr (72 samples)
Loading potomac2/1_year_drought…
  time 37.00 → 45.75 yr (36 samples)
Loading potomac2/3_year_drought…
  time 37.00 → 47.75 yr (44 samples)
Loading potomac2/10_year_drought…
  time 37.00 → 54.75 yr (72 samples)
Loading wolf2/short_baseline…
  time 37.00 → 54.75 yr (72 samples)
Loading wolf2/1_year_drought…
  time 37.00 → 45.75 yr (36 samples)
Loading wolf2/3_year_drought…
  time 37.00 → 47.75 yr (44 samples)
Loading wolf2/10_year_drought…
  time 37.00 → 54.75 yr (72 samples)
Loading wolf2/50_year_drought…


## Short droughts (1 / 3 / 10 yr) — absolute

One row per drought length; columns are storage and outlet flow. `short_baseline` is overlaid in grey on each panel.

In [ ]:
for domain_name, members in series.items():
    droughts = [m for m in SHORT_DROUGHTS if m in members]
    if not droughts:
        continue
    baseline = members.get("short_baseline")

    nrows = len(droughts)
    fig, axes = plt.subplots(
        nrows, 2, figsize=(11, 2.6 * nrows), sharex="col", squeeze=False
    )

    for row, member in enumerate(droughts):
        ds = members[member]
        ax_s, ax_f = axes[row]

        if baseline is not None:
            n = min(ds.sizes["time"], baseline.sizes["time"])
            baseline.storage.isel(time=slice(0, n)).plot(
                ax=ax_s, color="0.65", lw=1.2, label="short_baseline"
            )
            baseline.outlet_flow.isel(time=slice(0, n)).plot(
                ax=ax_f, color="0.65", lw=1.2, label="short_baseline"
            )

        ds.storage.plot(ax=ax_s, color="C0", lw=1.4, label=member)
        ds.outlet_flow.plot(ax=ax_f, color="C0", lw=1.4, label=member)

        mark_drought(ax_s, label=(row == 0))
        mark_drought(ax_f, label=(row == 0))

        ax_s.set_ylabel("storage")
        ax_f.set_ylabel("outlet flow")
        ax_s.set_title(f"{short_label(member)} — storage")
        ax_f.set_title(f"{short_label(member)} — outlet flow")
        ax_s.set_xlabel("")
        ax_f.set_xlabel("")

        if row == 0:
            ax_s.legend(fontsize=7, loc="best")

    axes[-1, 0].set_xlabel("Year (from sequence start)")
    axes[-1, 1].set_xlabel("Year (from sequence start)")
    fig.suptitle(
        f"{domain_name}: short droughts vs short_baseline",
        y=1.01,
    )
    plt.tight_layout()
    out = FIG_DIR / f"{domain_name}_short_droughts_absolute.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("wrote", out)
    plt.show()

## Short droughts — anomaly vs `short_baseline`

Same layout: one row per drought length.

In [ ]:
for domain_name, members in series.items():
    if "short_baseline" not in members:
        continue
    baseline = members["short_baseline"]
    droughts = [m for m in SHORT_DROUGHTS if m in members]

    nrows = len(droughts)
    fig, axes = plt.subplots(
        nrows, 2, figsize=(11, 2.6 * nrows), sharex="col", squeeze=False
    )

    for row, member in enumerate(droughts):
        ds = members[member]
        ax_s, ax_f = axes[row]
        n = min(ds.sizes["time"], baseline.sizes["time"])

        d_stor = ds.storage.isel(time=slice(0, n)) - baseline.storage.isel(
            time=slice(0, n)
        )
        d_flow = ds.outlet_flow.isel(time=slice(0, n)) - baseline.outlet_flow.isel(
            time=slice(0, n)
        )
        d_stor.plot(ax=ax_s, color="C0", lw=1.4)
        d_flow.plot(ax=ax_f, color="C0", lw=1.4)

        mark_drought(ax_s)
        mark_drought(ax_f)
        ax_s.axhline(0, color="k", lw=0.6)
        ax_f.axhline(0, color="k", lw=0.6)

        ax_s.set_ylabel("Δ storage")
        ax_f.set_ylabel("Δ outlet flow")
        ax_s.set_title(f"{short_label(member)} — Δ storage")
        ax_f.set_title(f"{short_label(member)} — Δ outlet flow")
        ax_s.set_xlabel("")
        ax_f.set_xlabel("")

    axes[-1, 0].set_xlabel("Year (from sequence start)")
    axes[-1, 1].set_xlabel("Year (from sequence start)")
    fig.suptitle(
        f"{domain_name}: drought − short_baseline",
        y=1.01,
    )
    plt.tight_layout()
    out = FIG_DIR / f"{domain_name}_short_droughts_anomaly.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("wrote", out)
    plt.show()

## 50-year drought — absolute

Separate figure so the long record does not crush the short-drought panels.

In [ ]:
member = "50_year_drought"
for domain_name, members in series.items():
    if member not in members:
        continue
    ds = members[member]

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ds.storage.plot(ax=axes[0], color="C0", lw=1.3)
    ds.outlet_flow.plot(ax=axes[1], color="C0", lw=1.3)

    for ax in axes:
        mark_drought(ax, label=True)

    axes[0].set_ylabel("Total subsurface storage")
    axes[0].set_title(f"{domain_name} {member}")
    axes[0].legend(fontsize=8)
    axes[1].set_ylabel("Outlet overland flow")
    axes[1].set_xlabel("Year (from sequence start)")

    plt.tight_layout()
    out = FIG_DIR / f"{domain_name}_50_year_drought_absolute.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("wrote", out)
    plt.show()

## 50-year drought — delta

Uses long `baseline` when available. Until that run finishes, falls back to change relative to the value at drought onset (year 40).

In [ ]:
member = "50_year_drought"
for domain_name, members in series.items():
    if member not in members:
        continue
    ds = members[member]
    baseline = members.get("baseline")

    if baseline is not None:
        n = min(ds.sizes["time"], baseline.sizes["time"])
        d_stor = ds.storage.isel(time=slice(0, n)) - baseline.storage.isel(
            time=slice(0, n)
        )
        d_flow = ds.outlet_flow.isel(time=slice(0, n)) - baseline.outlet_flow.isel(
            time=slice(0, n)
        )
        mode = "vs long baseline"
        suffix = "vs_baseline"
    else:
        d_stor = delta_from_onset(ds.storage)
        d_flow = delta_from_onset(ds.outlet_flow)
        mode = "from drought onset"
        suffix = "from_onset"
        print(
            f"{domain_name}: long baseline not loaded yet; "
            f"plotting Δ from drought onset"
        )

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    d_stor.plot(ax=axes[0], color="C0", lw=1.3)
    d_flow.plot(ax=axes[1], color="C0", lw=1.3)

    for ax in axes:
        mark_drought(ax, label=True)
        ax.axhline(0, color="k", lw=0.6)

    axes[0].set_ylabel("Δ storage")
    axes[0].set_title(f"{domain_name} {member} — Δ {mode}")
    axes[0].legend(fontsize=8)
    axes[1].set_ylabel("Δ outlet flow")
    axes[1].set_xlabel("Year (from sequence start)")

    plt.tight_layout()
    out = FIG_DIR / f"{domain_name}_50_year_drought_delta_{suffix}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print("wrote", out)
    plt.show()